# Generate Clutter Point Cloud sampples for classification
(x, y, z, v),
corrdinates are centralized

## Environment Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
MYDATASET_RADAR_PATH = "/content/drive/MyDrive/THESIS_dataset/mmw/MyDataset_rsu1/radar"
MYMODELNET_ClUTTER_PATH = "/content/drive/MyDrive/THESIS_dataset/mmw/MyModelNet_cls/clutter"

In [4]:
import sys
sys.path.append('/content/drive/MyDrive/THESIS/code')

import numpy as np
import os
from sklearn.cluster import DBSCAN

## Use DBSCAN to find clutter above the ground

In [7]:
def generate_clutter_samples(index, counter_clutter):
  radar_txt_path = os.path.join(MYDATASET_RADAR_PATH, index + ".txt")
  radar_points_labels = np.loadtxt(radar_txt_path, delimiter=',')
  unlabeled_points = radar_points_labels[radar_points_labels[:,-1] == -1]
  unlabeled_points = unlabeled_points[:, :4]

  # cluster using DBSCAN
  clustering = DBSCAN(eps=2.0, min_samples=2).fit(unlabeled_points[:, :3])
  dbscan_labels = clustering.labels_
  max_label = dbscan_labels.max()

  # find clutter above the ground with prob of 15%
  prob = 0.15
  for i in range(max_label):
      ps = unlabeled_points[dbscan_labels == i]

      if ps.size > 0 and 0.2 < np.mean(ps[:, 2], axis=0) < 3 and np.random.rand() < prob:
              centroid = np.mean(ps[:, :3], axis=0)
              ps[:, 0:3] = ps[:, 0:3] - centroid

              clutter_save_pth = os.path.join(MYMODELNET_ClUTTER_PATH, f"clutter_{counter_clutter:05d}.txt")
              np.savetxt(clutter_save_pth, ps, fmt='%.6f', delimiter=',')
              counter_clutter += 1

  return counter_clutter

In [8]:
index_range = ["016653", "017853"] # 017853
index = index_range[0]

counter_clutter = 1
while index != index_range[1]:
    counter_clutter = generate_clutter_samples(index, counter_clutter)

    index = f"{int(index) + 1:06d}"
    if counter_clutter % 100 == 0:
        print(index)

016708
016855
016910
016983
017079
017127
017205
017332
017354
017406
017407
017534
017583
017726
